# Stacking

This notebook implements a stacking ensemble learning technique (meta-learning) for fake news detection.

## Optimization Problem

MIN Z

$$
Z = - \frac{1}{N} \displaystyle\sum_{i=1}^{N} α_{c_i} [w_1 y_i log(F(x_i)) + w_0 (1 - y_i) log(1 - F(x_i))] + λ Ω(θ)
$$

<br>SUBJECT TO <br><br>

$C_1$: Stacking meta-learner

$$F(x) = g(f_1(x), f_2(x), ⋯ , f_M(x))$$

$C_2$: Feature Mapping

$$x_i = ϕ (title_i, text_i, category_i, dataset_i)$$

$C_3$: Binary Labels

$$y_i ∈ \{0, 1\}$$

$C_4$: Category Weight

$$α_{c} = \frac{1}{freq(c_i)}$$

$C_5$: Class Weight

$$w_1 = \frac{N}{2N_1}, \qquad w_0 = \frac{N}{2N_0}$$


## Colab Configuration

**Project Environment** : Run the code cell below to auto-detect data ingestion script. <br>
**Stand-alone Environment** : Skip the code cell below and upload the 'ingest_data.py' script from github.

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import sys
sys.path.append('/content/drive/MyDrive/project/COMP3608PROJECT')

Mounted at /content/drive


## Environment Configuration

The following code cell contains dependencies like third-party libraries that will be used in this notebook.

In [2]:
from ingest_data import load_datasets
import warnings
import re
import pandas as pd
import numpy as np
import scipy.sparse as sp
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import train_test_split
from sklearn.model_selection import StratifiedKFold
from sklearn.base import clone


## Data Ingestion

Load all fake news datasets using the shared 'ingest_data.py' script. <br>

The resulting dataframe `df` has five columns:
- `title` : title of news
- `text` : actual text content of news
- `category` : type of news
- `label` : 0 = fake, 1 = real
- `dataset` : source dataset of the news record

Duplicate and null text rows are removed during ingestion resulting in ~99K records across three datasets and eight original columns.

In [3]:
df = load_datasets()

------------------------------------------------------------
Fake News Dataset Ingestion
------------------------------------------------------------

Loading bhavikjikadara ...


100%|██████████| 18.1M/18.1M [00:00<00:00, 112MB/s] 

Extracting zip of true.csv...


[bhavik] Loaded 'true.csv': 21417 rows


100%|██████████| 22.9M/22.9M [00:00<00:00, 117MB/s]

...

100%|██████████| 32.8M/32.8M [00:00<00:00, 135MB/s]


[mahdi] Loaded 'fake_news_dataset.csv': 20000 rows

Loading shawkyelgendy ...


100%|██████████| 1.08M/1.08M [00:00<00:00, 94.1MB/s]

Extracting zip of real.csv...
[shawky] Loaded 'real.csv': 21869 rows


Using Colab cache for faster access to the 'fake-news-football' dataset.
[shawky] Loaded 'fake.csv': 19999 rows

Dropped 648 rows with empty/null text.
Dropped 6,650 duplicate rows.

------------------------------------------------------------
Fake News Dataset Summary
------------------------------------------------------------
Total rows: 99,468
Fake (0): 47,089
Real (1): 52,379

Rows per source:
shawkyelgendy          40,824
bhavikjikadara         38,644
mahdimashayekhi        20,000

Categories:
  Sports                 43,691
  Politics               21,635
  News                   19,811
  Health                 2,922
  Entertainment          2,889
  Technology             2,882
  Business               2,849
  Science                2,789
------------------------------------------------------------


## Text Cleaning and Preprocessing Pipeline

Raw news text contains noise that degrades quality of TF-IDF. <br>

We concatenate `title` and `text` into a single `content` field, then apply a cleaning pipeline that:
- Converts all text to lower case
- Removes URLs
- Strips HTML entities and HTML tags
- Removes punctuation and digits
- Collapses whitespace

Note: Stemming/Lemmatisation is NOT applied at this stage. TF-IDF witha sublinear term-frequency scale handles term frequency inflation naturally <br>

The cleaned text is stored in the new column `content`.

In [4]:
def clean_text(text: str) -> str:
    if not isinstance(text, str):
        return ""

    text = text.lower()
    text = re.sub(r'https?//S+|www\.\S+', ' ', text) # Remove URLs
    text = re.sub(r'<[^>]+>', ' ', text) # Strip HTML Tags
    text = re.sub(r'&[a-z]+;', ' ', text) # Strip HTML Entities
    text = re.sub(r'[^a-z\s]', ' ', text) # Keep letters only
    text = re.sub(r'\s+', ' ', text).strip() # Collapse whitespace
    return text

df['content'] = (df['title'].fillna('') + ' ' + df['text'].fillna('')).apply(clean_text)

df.head()


,title,text,label,category,dataset,content
0,"As U.S. budget fight looms, Republicans flip t...",WASHINGTON (Reuters) - The head of a conservat...,1,Politics,bhavikjikadara,as u s budget fight looms republicans flip the...
1,U.S. military to accept transgender recruits o...,WASHINGTON (Reuters) - Transgender people will...,1,Politics,bhavikjikadara,u s military to accept transgender recruits on...
2,Senior U.S. Republican senator: 'Let Mr. Muell...,WASHINGTON (Reuters) - The special counsel inv...,1,Politics,bhavikjikadara,senior u s republican senator let mr mueller d...
3,FBI Russia probe helped by Australian diplomat...,WASHINGTON (Reuters) - Trump campaign adviser ...,1,Politics,bhavikjikadara,fbi russia probe helped by australian diplomat...
4,Trump wants Postal Service to charge 'much mor...,SEATTLE/WASHINGTON (Reuters) - President Donal...,1,Politics,bhavikjikadara,trump wants postal service to charge much more...


## TF-IDF Feature Extraction

$C_2$: $x_i = ϕ(title_i, text_i, category_i, dataset_i)$

Constraint $C_2$ maps raw inputs to a feature vector. The primary signal comes from TF-IDF on `content`. Use:

<pre>
sublinear_tf = True               - Dampens high-frequency terms
max_features = 100 000            - Vocabulary cap to control dimensionality
ngram_range = (1,2)               - unigrams + bigrams to capture short phrases
min_df = 3                        - ignore very rare terms
strip_accents / analyzer = 'word' - standard tokenization
</pre>

The vectorizer is fitted only on the training split to prevent data leakage. At this stage fit using the full dataframe to obtain the index. The fit is to be redone after train/test splitting.

In [5]:
TFIDF_PARAMS = {
    'sublinear_tf': True,
    'max_features': 100000,
    'ngram_range': (1, 2),
    'min_df': 3,
    'strip_accents': 'unicode',
    'analyzer': 'word',
    'token_pattern': r'\b[a-z]{2,}\b' # min 2 characters per word
}

tfidf = TfidfVectorizer(**TFIDF_PARAMS)

print("TF-IDF Vectorizer Configured")
print(f"    ngram_range  : {tfidf.ngram_range}")
print(f"    max_features : {tfidf.max_features:,}")
print(f"    sublinear_tf : {tfidf.sublinear_tf}")

TF-IDF Vectorizer Configured
    ngram_range  : (1, 2)
    max_features : 100,000
    sublinear_tf : True


## Categorical Encoding

The `category` and `dataset` columns are categorical. <br>
One-hot encode these columns and append them as sparse columns to the TF-IDF matrix so the full feature vector satisfies $C_2$. <br>
One-hot encoding is chosen over ordinal encoding because there is no inherent ordering among categories. <br>
We use `sparse_output=True` to keep memory usage manageable when concatenating with the TF-IDF matrix.

In [6]:
cat_encoder = OneHotEncoder(sparse_output=True, handle_unknown='ignore')

# Fit on full data, Transform post split
cat_features_all = cat_encoder.fit_transform(df[['category', 'dataset']])

print("Categorical Encoder Fitted")
print(f"    Input columns   : category, dataset")
print(f"    Output shape    : {cat_features_all.shape}")
print(f"    Category values : {cat_encoder.categories_[0].tolist()}")
print(f"    Dataset values  : {cat_encoder.categories_[1].tolist()}")

Categorical Encoder Fitted
    Input columns   : category, dataset
    Output shape    : (99468, 11)
    Category values : ['Business', 'Entertainment', 'Health', 'News', 'Politics', 'Science', 'Sports', 'Technology']
    Dataset values  : ['bhavikjikadara', 'mahdimashayekhi', 'shawkyelgendy']


## Sample Weight Computation

The objective function $Z$ uses two complementary weight terms: <br><br>
**Category Weights** ($C_4$): Adjusts weights for over-represented categories (e.g. Sports = 44% of data) so that rarer categories (e.g. Science, Business) contributes proportionally.

$$α_c = \frac{1}{freq(c_i)}$$

**Class Weights** ($C_5$): Corrects for class imbalance between real and fake news.

$$w_1 = \frac{N}{2N_1}, \qquad w_0 = \frac{N}{2N_0}$$

The final per-sample weight is the product $w_{y_i} ⋅ α_{c_i}$, which is passed as `sample_weight` to each base learner and the meta-learner.

In [7]:
# --- Category weights ---
# C4: alpha_c = 1 / freq(c)
cat_freq = df['category'].value_counts(normalize=True) # Relative Frequency
category_weight_per_sample = df['category'].map(lambda c: 1.0 / cat_freq[c]).values

# --- Class weights ---
N = len(df)
label_counts = df['label'].value_counts()
N0 = label_counts[0]
N1 = label_counts[1]

# C5: w_i = N / (2 * N_i)
w0 = N / (2 * N0)
w1 = N / (2 * N1)

class_weight_map = {0: w0, 1: w1}
class_weight_per_sample = df['label'].map(class_weight_map).values

# --- Final sample weight ---
sample_weights = class_weight_per_sample * category_weight_per_sample # Combine sample weights
sample_weights = sample_weights / sample_weights.mean() # Normalise so weights sum to N

df['sample_weight'] = sample_weights

print(f"Class weights")
print(f"    w0 (fake) : {w0:.4f}")
print(f"    w1 (real) : {w1:.4f}")
print(f"\nCategory weight")
for cat, freq in cat_freq.items():
    print(f"    {cat:<20s}: alpha = {1/freq:.4f}  (freq = {freq:.3f})")
print(f"\nSample weights stats")
print(f"    min  : {sample_weights.min():.4f}")
print(f"    max  : {sample_weights.max():.4f}")
print(f"    mean : {sample_weights.mean():.4f}")

Class weights
    w0 (fake) : 1.0562
    w1 (real) : 0.9495

Category weight
    Sports              : alpha = 2.2766  (freq = 0.439)
    Politics            : alpha = 4.5976  (freq = 0.218)
    News                : alpha = 5.0208  (freq = 0.199)
    Health              : alpha = 34.0411  (freq = 0.029)
    Entertainment       : alpha = 34.4299  (freq = 0.029)
    Technology          : alpha = 34.5135  (freq = 0.029)
    Business            : alpha = 34.9133  (freq = 0.029)
    Science             : alpha = 35.6644  (freq = 0.028)

Sample weights stats
    min  : 0.2698
    max  : 4.7007
    mean : 1.0000


## Stratified Train/Test Split

Train data will comprise of 80% of the data. <br>
Test data will comprise of 20% of the data. Test data is never touched during model training or OOF generation. <br>

Stratification is performed jointly on `label` and `category` to ensure both class and category distributions are preserved in both splits. This prevents distribution shift between training and evaluation. <br>

After splitting, the TF-IDF vectorizer is fitted only on training data and used to transform both splits preventing any leakage of test vocabulary into feature weights.

In [8]:
# Create a joint stratification key: label + category
strat_key = df['label'].astype(str) + '_' + df['category']

idx = np.arange(len(df))
idx_train, idx_test = train_test_split(
    idx,
    test_size=0.20,
    random_state=42,
    stratify=strat_key
)

df_train = df.iloc[idx_train].reset_index(drop=True)
df_test = df.iloc[idx_test].reset_index(drop=True)
sw_train = sample_weights[idx_train]
sw_test = sample_weights[idx_test]

# Fit TF-IDF on train only, transform both splits
X_train_tfidf = tfidf.fit_transform(df_train['content'])
X_test_tfidf = tfidf.transform(df_test['content'])

# Categorical features - encoder already fitted on full data
X_train_cat = cat_encoder.transform(df_train[['category', 'dataset']])
X_test_cat = cat_encoder.transform(df_test[['category', 'dataset']])

# Full feature matrices
X_train = sp.hstack([X_train_tfidf, X_train_cat], format='csr')
X_test = sp.hstack([X_test_tfidf, X_test_cat], format='csr')

Y_train = df_train['label'].values
Y_test = df_test['label'].values

print(f"Train size : {len(df_train):>7,}")
print(f"Test size  : {len(df_test):>6,}")
print(f"X_train    : {X_train.shape}")
print(f"X_test     : {X_test.shape}")
print(f"\nTrain label distribution")
print(f"    Fake: {(Y_train==0).sum():,}")
print(f"    Real: {(Y_train==1).sum():,}")
print(f"\nTest label distribution")
print(f"    Fake: {(Y_test==0).sum():,}")
print(f"    Real: {(Y_test==1).sum():,}")

Train size :  79,574
Test size  : 19,894
X_train    : (79574, 100011)
X_test     : (19894, 100011)

Train label distribution
    Fake: 37,671
    Real: 41,903

Test label distribution
    Fake: 9,418
    Real: 10,476


## K-Fold OOF Framework

In [ ]:
warnings.filterwarnings('ignore')

N_SPLITS = 5
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=42)
